# Multi-Agent Research Assistant Demo
Runs the five-agent research workflow and prints the execution trace.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from coordinator import Coordinator
question = "What are benefits and limitations of AI in education?"
result = Coordinator().run(question)
print('Status:', result['status'])
print('Semantic Kernel enabled:', result['semantic_kernel_enabled'])
print('Sources:', [x['source_id'] for x in result['evidence']])

Status: success
Semantic Kernel enabled: True
Sources: ['paper_05', 'paper_02', 'paper_03', 'paper_01', 'paper_04']


In [2]:
for event in result['trace']:
    print(f"{event['sender']} -> {event['recipient']} | {event['message_type']} | {event['status']} | attempt={event['metadata'].get('attempt', 1)}")

coordinator -> planner_agent | task | success | attempt=1
planner_agent -> research_agent | agent_result | success | attempt=1
coordinator -> research_agent | task | success | attempt=1
research_agent -> analysis_agent | agent_result | success | attempt=1
coordinator -> analysis_agent | task | success | attempt=1
analysis_agent -> reviewer_agent | agent_result | success | attempt=1
coordinator -> reviewer_agent | task | success | attempt=1
reviewer_agent -> coordinator | review | success | attempt=1
coordinator -> writer_agent | task | success | attempt=1
writer_agent -> user | agent_result | success | attempt=1


In [3]:
print(result['report'] or f"No report generated: {result['status']}")

# Multi-Agent Research Brief

**Research question:** What are benefits and limitations of AI in education?

## Executive summary
- **Claim:** The guidance proposes key steps for regulating generative AI, including protection of data privacy and age-appropriate approaches. [paper_05]
  - **Retrieved corpus text:** > The guidance proposes key steps for regulating generative AI, including protection of data privacy and age-appropriate approaches.
- **Claim:** This systematic review mapped the thematic landscape of 407 publications indexed in Web of Science, ScienceDirect and Scopus. [paper_02]
  - **Retrieved corpus text:** > This systematic review mapped the thematic landscape of 407 publications indexed in Web of Science, ScienceDirect and Scopus.
- **Claim:** This systematic review attempts to render an evidence-based understanding of the interaction between artificial intelligence and science education. [paper_03]
  - **Retrieved corpus text:** > This systematic review attempts to ren

## Failure recovery demo
Inject one transient planner failure and observe the retry in the trace.

In [4]:
def fail_once(agent_name, attempt):
    return agent_name == 'planner_agent' and attempt == 1
retry_result = Coordinator(failure_injector=fail_once).run('How can AI support teachers and students?')
print('Status:', retry_result['status'])
for event in retry_result['trace']:
    if event['message_type'] == 'error' or event['payload'].get('event') == 'retry_scheduled':
        print(event['sender'], event['status'], event['payload'], event['error'])

Status: success
planner_agent failed {} Injected failure for planner_agent, attempt 1
coordinator success {'event': 'retry_scheduled'} None
